In [2]:
!pip install -U bitsandbytes transformers peft accelerate datasets scipy einops evaluate trl rouge_score -qq

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 6.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 124.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 54.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 50.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.2/35.2 MB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 697.4/697.4 kB 57.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 16.5 MB/s eta 0:00:00


In [3]:
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    HfArgumentParser,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    GenerationConfig
)
from tqdm import tqdm
from trl import SFTTrainer
import torch
import time
import pandas as pd
import numpy as np
# from huggingface_hub import interpreter_login

# interpreter_login()

In [4]:
import os
# disable Weights and Biases
os.environ['WANDB_DISABLED']="true"

In [5]:
from pynvml import *

def print_gpu_utilization():
    nvmlInit()
    handle = nvmlDeviceGetHandleByIndex(0)
    info = nvmlDeviceGetMemoryInfo(handle)
    print(f"GPU memory occupied: {info.used//1024**2} MB.")

# Load dataset

In [6]:
huggingface_dataset_name = "neil-code/dialogsum-test"
dataset = load_dataset(huggingface_dataset_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

train.csv: 0.00B [00:00, ?B/s]

validation.csv: 0.00B [00:00, ?B/s]

test.csv: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/1999 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/499 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/499 [00:00<?, ? examples/s]

In [7]:
dataset['train'][0]

{'id': 'train_0',
 'dialogue': "#Person1#: Hi, Mr. Smith. I'm Doctor Hawkins. Why are you here today?\n#Person2#: I found it would be a good idea to get a check-up.\n#Person1#: Yes, well, you haven't had one for 5 years. You should have one every year.\n#Person2#: I know. I figure as long as there is nothing wrong, why go see the doctor?\n#Person1#: Well, the best way to avoid serious illnesses is to find out about them early. So try to come at least once a year for your own good.\n#Person2#: Ok.\n#Person1#: Let me see here. Your eyes and ears look fine. Take a deep breath, please. Do you smoke, Mr. Smith?\n#Person2#: Yes.\n#Person1#: Smoking is the leading cause of lung cancer and heart disease, you know. You really should quit.\n#Person2#: I've tried hundreds of times, but I just can't seem to kick the habit.\n#Person1#: Well, we have classes and some medications that might help. I'll give you more information before you leave.\n#Person2#: Ok, thanks doctor.",
 'summary': "Mr. Smith'

In [8]:
compute_dtype = getattr(torch, "float16")
bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=compute_dtype,
        bnb_4bit_use_double_quant=False,
    )

# Load base model

In [9]:
model_name='microsoft/phi-2'
device_map = {"": 0}
original_model = AutoModelForCausalLM.from_pretrained(model_name,
                                                      device_map=device_map,
                                                      quantization_config=bnb_config,
                                                      trust_remote_code=True,
                                                      )

tokenizer = AutoTokenizer.from_pretrained(model_name,
                                          trust_remote_code=True,
                                          padding_side="left",
                                          add_eos_token=True,add_bos_token=True,
                                          use_fast=False)
tokenizer.pad_token = tokenizer.eos_token

config.json:   0%|          | 0.00/735 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/453 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

In [10]:
eval_tokenizer = AutoTokenizer.from_pretrained(model_name, add_bos_token=True, trust_remote_code=True, use_fast=False)
eval_tokenizer.pad_token = eval_tokenizer.eos_token

def gen(model,p, maxlen=100, sample=True):
    toks = eval_tokenizer(p, return_tensors="pt")
    res = model.generate(**toks.to("cuda"), max_new_tokens=maxlen, do_sample=sample,num_return_sequences=1,temperature=0.1,num_beams=1,top_p=0.95,).to('cpu')
    return eval_tokenizer.batch_decode(res,skip_special_tokens=True)

# Test model with zero-shot inference

In [11]:
%%time
from transformers import set_seed
seed = 42
set_seed(seed)

index = 10

prompt = dataset['test'][index]['dialogue']
summary = dataset['test'][index]['summary']

formatted_prompt = f"Instruct: Summarize the following conversation.\n{prompt}\nOutput:\n"
res = gen(original_model,formatted_prompt,100,)
#print(res[0])
output = res[0].split('Output:\n')[1]

dash_line = '-'.join('' for x in range(100))
print(dash_line)
print(f'INPUT PROMPT:\n{formatted_prompt}')
print(dash_line)
print(f'BASELINE HUMAN SUMMARY:\n{summary}\n')
print(dash_line)
print(f'MODEL GENERATION - ZERO SHOT:\n{output}')

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


---------------------------------------------------------------------------------------------------
INPUT PROMPT:
Instruct: Summarize the following conversation.
#Person1#: Happy Birthday, this is for you, Brian.
#Person2#: I'm so happy you remember, please come in and enjoy the party. Everyone's here, I'm sure you have a good time.
#Person1#: Brian, may I have a pleasure to have a dance with you?
#Person2#: Ok.
#Person1#: This is really wonderful party.
#Person2#: Yes, you are always popular with everyone. and you look very pretty today.
#Person1#: Thanks, that's very kind of you to say. I hope my necklace goes with my dress, and they both make me look good I feel.
#Person2#: You look great, you are absolutely glowing.
#Person1#: Thanks, this is a fine party. We should have a drink together to celebrate your birthday
Output:

---------------------------------------------------------------------------------------------------
BASELINE HUMAN SUMMARY:
#Person1# attends Brian's birthday pa

# Preprocess dataset

In [12]:
def create_prompt_formats(sample):
    """
    Format various fields of the sample ('instruction','output')
    Then concatenate them using two newline characters
    :param sample: Sample dictionnary
    """
    INTRO_BLURB = "Below is an instruction that describes a task. Write a response that appropriately completes the request."
    INSTRUCTION_KEY = "### Instruct: Summarize the below conversation."
    RESPONSE_KEY = "### Output:"
    END_KEY = "### End"

    blurb = f"\n{INTRO_BLURB}"
    instruction = f"{INSTRUCTION_KEY}"
    input_context = f"{sample['dialogue']}" if sample["dialogue"] else None
    response = f"{RESPONSE_KEY}\n{sample['summary']}"
    end = f"{END_KEY}"

    parts = [part for part in [blurb, instruction, input_context, response, end] if part]

    formatted_prompt = "\n\n".join(parts)
    sample["text"] = formatted_prompt

    return sample

In [13]:
# SOURCE https://github.com/databrickslabs/dolly/blob/master/training/trainer.py
def get_max_length(model):
    conf = model.config
    max_length = None
    for length_setting in ["n_positions", "max_position_embeddings", "seq_length"]:
        max_length = getattr(model.config, length_setting, None)
        if max_length:
            print(f"Found max lenth: {max_length}")
            break
    if not max_length:
        max_length = 1024
        print(f"Using default max length: {max_length}")
    return max_length


def preprocess_batch(batch, tokenizer, max_length):
    """
    Tokenizing a batch
    """
    return tokenizer(
        batch["text"],
        max_length=max_length,
        truncation=True,
    )

In [14]:
from functools import partial

# SOURCE https://github.com/databrickslabs/dolly/blob/master/training/trainer.py
def preprocess_dataset(tokenizer: AutoTokenizer, max_length: int,seed, dataset):
    """Format & tokenize it so it is ready for training
    :param tokenizer (AutoTokenizer): Model Tokenizer
    :param max_length (int): Maximum number of tokens to emit from tokenizer
    """

    # Add prompt to each sample
    print("Preprocessing dataset...")
    dataset = dataset.map(create_prompt_formats)#, batched=True)

    _preprocessing_function = partial(preprocess_batch, max_length=max_length, tokenizer=tokenizer)
    dataset = dataset.map(
        _preprocessing_function,
        batched=True,
        remove_columns=['id', 'topic', 'dialogue', 'summary'],
    )

    # Filter out samples that have input_ids exceeding max_length
    dataset = dataset.filter(lambda sample: len(sample["input_ids"]) < max_length)

    # Shuffle dataset
    dataset = dataset.shuffle(seed=seed)

    return dataset

In [15]:
print_gpu_utilization()

GPU memory occupied: 2630 MB.


In [16]:
# ## Pre-process dataset
max_length = get_max_length(original_model)
print(max_length)

train_dataset = preprocess_dataset(tokenizer, max_length,seed, dataset['train'])
eval_dataset = preprocess_dataset(tokenizer, max_length,seed, dataset['validation'])

Found max lenth: 2048
2048
Preprocessing dataset...


Map:   0%|          | 0/1999 [00:00<?, ? examples/s]

Map:   0%|          | 0/1999 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1999 [00:00<?, ? examples/s]

Preprocessing dataset...


Map:   0%|          | 0/499 [00:00<?, ? examples/s]

Map:   0%|          | 0/499 [00:00<?, ? examples/s]

Filter:   0%|          | 0/499 [00:00<?, ? examples/s]

In [17]:
print(f"Shapes of the datasets:")
print(f"Training: {train_dataset.shape}")
print(f"Validation: {eval_dataset.shape}")
print(train_dataset)

Shapes of the datasets:
Training: (1999, 3)
Validation: (499, 3)
Dataset({
    features: ['text', 'input_ids', 'attention_mask'],
    num_rows: 1999
})


In [18]:
train_dataset['text'][1]

"\nBelow is an instruction that describes a task. Write a response that appropriately completes the request.\n\n### Instruct: Summarize the below conversation.\n\n#Person1#: Let' s got out tomorrow night. We can go to a bar and try to find you a girlfriend. \n#Person2#: I don' t think that' s a good idea. I am just not good with approaching someone and starting up a conversation. \n#Person1#: Maybe you just need a few pick-up lines, you know, break the ice. \n#Person2#: Pick-up lines don' t work! \n#Person1#: Come on! You can just walk up to a girl and say'If you were a booger I' d pick you first. ' \n#Person2#: What? Come on! That's just lame! No girl would fall for that! \n#Person1#: Fine, then you can say, 'So there you are! I' ve been looking all over for YOU, the woman of my dreams! ' \n#Person2#: That' s a good one! I think that' s pretty funny. \n#Person1#: Yeah, so you make her laugh, you make a fool of yourself a little bit and then you buy her a drink. \n#Person2#: Ok, how do

# Set up parameter-efficient fine-tuning experiments

This notebook is now organized to answer the assignment directly:

1. **Vanilla Phi-2** baseline summarization (already evaluated above)
2. **Default LoRA PEFT** with target modules `q_proj`, `k_proj`, `v_proj`, `dense`
3. **Q/V-only LoRA PEFT** with target modules `q_proj`, `v_proj`

For each fine-tuned model, the notebook records:
- trainable parameter count
- fine-tuning time
- ROUGE scores
- improvement over the vanilla Phi-2 baseline


In [19]:

def print_number_of_trainable_model_parameters(model):
    trainable_model_params = 0
    all_model_params = 0
    for _, param in model.named_parameters():
        all_model_params += param.numel()
        if param.requires_grad:
            trainable_model_params += param.numel()
    percentage = 100 * trainable_model_params / all_model_params
    summary = (
        f"trainable model parameters: {trainable_model_params}\n"
        f"all model parameters: {all_model_params}\n"
        f"percentage of trainable model parameters: {percentage:.4f}%"
    )
    return summary, trainable_model_params, all_model_params, percentage


In [20]:

# Experiment controls
base_model_id = "microsoft/phi-2"
SEED = 42
MAX_STEPS = 500          #changed from 1000 to 500
EVAL_SAMPLE_SIZE = 10
BASE_OUTPUT_DIR = "./phi2_lora_outputs"

os.makedirs(BASE_OUTPUT_DIR, exist_ok=True)

DEFAULT_TARGET_MODULES = ['q_proj', 'k_proj', 'v_proj', 'dense']
QV_ONLY_TARGET_MODULES = ['q_proj', 'v_proj']

EXPERIMENTS = {
    "default_lora": DEFAULT_TARGET_MODULES,
    "qv_only_lora": QV_ONLY_TARGET_MODULES,
}


In [21]:
import transformers

from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
import evaluate
import json

def load_quantized_base_model(model_id=base_model_id):
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        device_map="auto",
        quantization_config=bnb_config,
        trust_remote_code=True,

    )
    return model

def build_peft_model(target_modules, model_id=base_model_id):
    model = load_quantized_base_model(model_id)
    model.gradient_checkpointing_enable()
    model = prepare_model_for_kbit_training(model)

    config = LoraConfig(
        r=32,
        lora_alpha=32,
        target_modules=target_modules,
        bias="none",
        lora_dropout=0.05,
        task_type="CAUSAL_LM",
    )

    peft_model = get_peft_model(model, config)
    return peft_model, config

def build_trainer(model, output_dir):
    training_args = TrainingArguments(
        output_dir=output_dir,
        warmup_steps=1,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        max_steps=MAX_STEPS,
        learning_rate=2e-4,
        optim="paged_adamw_8bit",
        logging_steps=25,
        logging_dir=os.path.join(output_dir, "logs"),
        save_strategy="steps",
        save_steps=50,
        eval_strategy="steps",
        eval_steps=50,
        do_eval=True,
        gradient_checkpointing=True,
        report_to="none",
        seed=SEED,
    )

    model.config.use_cache = False

    trainer = Trainer(
        model=model,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        args=training_args,
        data_collator=transformers.DataCollatorForLanguageModeling(tokenizer, mlm=False),
    )
    return trainer, training_args

def generate_summary(model, dialogue, max_new_tokens=100):
    prompt = f"Instruct: Summarize the following conversation.\n{dialogue}\nOutput:\n"
    response = gen(model, prompt, max_new_tokens)[0]
    output = response.split("Output:\n", 1)[1] if "Output:\n" in response else response
    output = output.split("### End", 1)[0]
    output = output.split("#End", 1)[0]
    return output.strip()

def evaluate_model_rouge(model, dataset_obj, sample_size=EVAL_SAMPLE_SIZE):
    dialogues = dataset_obj['test'][0:sample_size]['dialogue']
    references = dataset_obj['test'][0:sample_size]['summary']

    predictions = []
    for dialogue in dialogues:
        predictions.append(generate_summary(model, dialogue))

    rouge = evaluate.load("rouge")
    rouge_scores = rouge.compute(
        predictions=predictions,
        references=references,
        use_aggregator=True,
        use_stemmer=True,
    )
    return predictions, references, rouge_scores

def run_lora_experiment(experiment_name, target_modules):
    output_dir = os.path.join(BASE_OUTPUT_DIR, experiment_name)

    peft_model, lora_config = build_peft_model(target_modules)
    param_summary, trainable_params, all_params, trainable_pct = print_number_of_trainable_model_parameters(peft_model)
    print(param_summary)

    trainer, training_args = build_trainer(peft_model, output_dir)
    print(f"Training device: {training_args.device}")

    start_time = time.time()
    train_result = trainer.train()
    fine_tune_seconds = time.time() - start_time

    adapter_path = trainer.state.best_model_checkpoint or trainer.state.global_step
    trainer.save_model(output_dir)

    del trainer
    torch.cuda.empty_cache()

    base_model_for_eval = load_quantized_base_model()
    ft_model = PeftModel.from_pretrained(
        base_model_for_eval,
        output_dir,
        torch_dtype=torch.float16,
        is_trainable=False
    )

    predictions, references, rouge_scores = evaluate_model_rouge(ft_model, dataset, sample_size=EVAL_SAMPLE_SIZE)

    results = {
        "experiment_name": experiment_name,
        "target_modules": target_modules,
        "trainable_params": int(trainable_params),
        "all_params": int(all_params),
        "trainable_pct": float(trainable_pct),
        "fine_tune_seconds": float(fine_tune_seconds),
        "fine_tune_minutes": float(fine_tune_seconds / 60.0),
        "max_steps": int(MAX_STEPS),
        "eval_sample_size": int(EVAL_SAMPLE_SIZE),
        "rouge": rouge_scores,
        "output_dir": output_dir,
    }

    with open(os.path.join(output_dir, "results.json"), "w") as f:
        json.dump(results, f, indent=2)

    return ft_model, results, predictions, references


In [22]:

# Sanity check: this is the vanilla quantized Phi-2 parameter count before LoRA
original_model_for_count = load_quantized_base_model()
base_param_summary, base_trainable, base_all, base_pct = print_number_of_trainable_model_parameters(original_model_for_count)
print(base_param_summary)
del original_model_for_count
torch.cuda.empty_cache()


Loading weights:   0%|          | 0/453 [00:00<?, ?it/s]

trainable model parameters: 263101440
all model parameters: 1521392640
percentage of trainable model parameters: 17.2935%


## Fine-tune experiment A: default LoRA (`q_proj`, `k_proj`, `v_proj`, `dense`)

In [23]:
import time
start_time = time.time()

default_ft_model, default_results, default_predictions, default_references = run_lora_experiment(
    "default_lora",
    DEFAULT_TARGET_MODULES
)

print(json.dumps(default_results, indent=2))


end_time = time.time()
print(f"Total training time: {end_time - start_time:.2f} seconds") #record finetuning time


Loading weights:   0%|          | 0/453 [00:00<?, ?it/s]

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


trainable model parameters: 20971520
all model parameters: 1542364160
percentage of trainable model parameters: 1.3597%
Training device: cuda:0


Step,Training Loss,Validation Loss
50,1.366374,1.359604
100,1.368042,1.348649
150,1.394273,1.338733
200,1.309104,1.334765
250,1.284512,1.331657
300,1.328288,1.329364
350,1.338800,1.326477
400,1.277754,1.325662
450,1.377686,1.324555
500,1.324796,1.323695


Loading weights:   0%|          | 0/453 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


{
  "experiment_name": "default_lora",
  "target_modules": [
    "q_proj",
    "k_proj",
    "v_proj",
    "dense"
  ],
  "trainable_params": 20971520,
  "all_params": 1542364160,
  "trainable_pct": 1.3596996444730667,
  "fine_tune_seconds": 2870.0404999256134,
  "fine_tune_minutes": 47.83400833209355,
  "max_steps": 500,
  "eval_sample_size": 10,
  "rouge": {
    "rouge1": 0.3647563670215904,
    "rouge2": 0.12211551219789996,
    "rougeL": 0.2974155404898105,
    "rougeLsum": 0.29834569491185026
  },
  "output_dir": "./phi2_lora_outputs/default_lora"
}
Total training time: 2954.51 seconds


In [24]:
#load vanilla phi2 model
original_model = load_quantized_base_model()
original_model_predictions, original_references, original_model_results = evaluate_model_rouge(
    original_model,
    dataset,
    sample_size=EVAL_SAMPLE_SIZE
)

print("ORIGINAL MODEL:")
print(original_model_results)


Loading weights:   0%|          | 0/453 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


ORIGINAL MODEL:
{'rouge1': np.float64(0.2888761467676735), 'rouge2': np.float64(0.10361408534161379), 'rougeL': np.float64(0.20955558279527847), 'rougeLsum': np.float64(0.22022167880792007)}


In [25]:
#default model predictions
#supposed to be comparison with vanilla and Q/V only model but i had to do these on two separate google accounts
import pandas as pd

comparison_df = pd.DataFrame({
    "default_lora_summary": default_predictions
})

comparison_df


,default_lora_summary
0,#Person1# asks Ms. Dawson to take a dictation ...
1,#Person1# asks Ms. Dawson to take a dictation ...
2,#Person1# asks Ms. Dawson to take a dictation ...
3,#Person2# got stuck in traffic again and #Pers...
4,#Person2# got stuck in traffic again and #Pers...
5,#Person2# got stuck in traffic again and #Pers...
6,Masha and Hero are getting divorced. Masha and...
7,Masha and Hero are getting divorced. Masha and...
8,Masha and Hero are getting divorced. Masha and...
9,#Person1# brings a gift for Brian's birthday a...


In [26]:
#compute ROUGE score for default only
all_rouge_df = pd.DataFrame([

    {"model": "default_lora", **default_results["rouge"]}
])

all_rouge_df


,model,rouge1,rouge2,rougeL,rougeLsum
0,default_lora,0.364756,0.122116,0.297416,0.298346


In [29]:
#compute improvement over vanilla phi 2
def rouge_improvement_df(baseline_scores, candidate_scores, candidate_name):
    rows = []
    for metric in baseline_scores.keys():
        baseline_value = baseline_scores[metric]
        candidate_value = candidate_scores[metric]
        rows.append({
            "model": candidate_name,
            "metric": metric,
            "baseline_score": baseline_value,
            "candidate_score": candidate_value,
            "absolute_improvement": candidate_value - baseline_value,
            "relative_improvement_pct": ((candidate_value - baseline_value) / baseline_value * 100.0) if baseline_value != 0 else np.nan,
        })
    return pd.DataFrame(rows)

default_improvement_df = rouge_improvement_df(original_model_results, default_results["rouge"], "default_lora")

# pd.concat([default_improvement_df], ignore_index=True)


## Quick qualitative example

In [30]:

from transformers import set_seed
set_seed(SEED)

index = 10
dialogue = dataset['test'][index]['dialogue']
summary = dataset['test'][index]['summary']

print("-" * 100)
print("INPUT DIALOGUE:")
print(dialogue)
print("-" * 100)
print("HUMAN SUMMARY:")
print(summary)
print("-" * 100)
print("VANILLA PHI-2 SUMMARY:")
print(generate_summary(original_model, dialogue))
print("-" * 100)
print("DEFAULT LORA SUMMARY:")
print(generate_summary(default_ft_model, dialogue))


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


----------------------------------------------------------------------------------------------------
INPUT DIALOGUE:
#Person1#: Happy Birthday, this is for you, Brian.
#Person2#: I'm so happy you remember, please come in and enjoy the party. Everyone's here, I'm sure you have a good time.
#Person1#: Brian, may I have a pleasure to have a dance with you?
#Person2#: Ok.
#Person1#: This is really wonderful party.
#Person2#: Yes, you are always popular with everyone. and you look very pretty today.
#Person1#: Thanks, that's very kind of you to say. I hope my necklace goes with my dress, and they both make me look good I feel.
#Person2#: You look great, you are absolutely glowing.
#Person1#: Thanks, this is a fine party. We should have a drink together to celebrate your birthday
----------------------------------------------------------------------------------------------------
HUMAN SUMMARY:
#Person1# attends Brian's birthday party. Brian thinks #Person1# looks great and charming.
--------

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Person1 and Person2 are at a party, and Person1 asks if they can have a dance. Person2 agrees and compliments Person1 on their appearance. Person1 thanks them and expresses their happiness with the party. Person2 agrees that it's a great party and suggests having a drink to celebrate.
----------------------------------------------------------------------------------------------------
DEFAULT LORA SUMMARY:
Brian's birthday party is going well and #Person1# compliments Brian's appearance. #Person1# invites Brian to have a dance and they both have a drink together to celebrate.


## Training time and trainable-parameter comparison

In [31]:

runtime_df = pd.DataFrame([
    {
        "model": "vanilla_phi2",
        "trainable_params": base_trainable,
        "trainable_pct": base_pct,
        "fine_tune_seconds": 0.0,
        "fine_tune_minutes": 0.0,
        "target_modules": "None (no fine-tuning)",
    },
    {
        "model": "default_lora",
        "trainable_params": default_results["trainable_params"],
        "trainable_pct": default_results["trainable_pct"],
        "fine_tune_seconds": default_results["fine_tune_seconds"],
        "fine_tune_minutes": default_results["fine_tune_minutes"],
        "target_modules": ", ".join(default_results["target_modules"]),
    }
    # {
    #     "model": "qv_only_lora",
    #     "trainable_params": qv_results["trainable_params"],
    #     "trainable_pct": qv_results["trainable_pct"],
    #     "fine_tune_seconds": qv_results["fine_tune_seconds"],
    #     "fine_tune_minutes": qv_results["fine_tune_minutes"],
    #     "target_modules": ", ".join(qv_results["target_modules"]),
    # },
])

runtime_df


,model,trainable_params,trainable_pct,fine_tune_seconds,fine_tune_minutes,target_modules
0,vanilla_phi2,263101440,17.293461,0.0000,0.000000,None (no fine-tuning)
1,default_lora,20971520,1.359700,2870.0405,47.834008,"q_proj, k_proj, v_proj, dense"


In [33]:

all_rouge_df.to_csv(os.path.join(BASE_OUTPUT_DIR, "rouge_scores.csv"), index=False)
runtime_df.to_csv(os.path.join(BASE_OUTPUT_DIR, "runtime_and_params.csv"), index=False)
comparison_df.to_csv(os.path.join(BASE_OUTPUT_DIR, "sample_predictions.csv"), index=False)

summary_report = {
    "baseline_rouge": original_model_results,
    "default_lora": default_results,
    # "qv_only_lora": qv_results,
}

with open(os.path.join(BASE_OUTPUT_DIR, "assignment_q2_summary.json"), "w") as f:
    json.dump(summary_report, f, indent=2)

print(f"Saved outputs to: {BASE_OUTPUT_DIR}")


Saved outputs to: ./phi2_lora_outputs


In [34]:

print("Final ROUGE table")
display(all_rouge_df)

print("\nFinal runtime / parameter table")
display(runtime_df)


Final ROUGE table


,model,rouge1,rouge2,rougeL,rougeLsum
0,default_lora,0.364756,0.122116,0.297416,0.298346



Final runtime / parameter table


,model,trainable_params,trainable_pct,fine_tune_seconds,fine_tune_minutes,target_modules
0,vanilla_phi2,263101440,17.293461,0.0000,0.000000,None (no fine-tuning)
1,default_lora,20971520,1.359700,2870.0405,47.834008,"q_proj, k_proj, v_proj, dense"


In [35]:

for _, row in runtime_df.iterrows():
    print(f"{row['model']}:")
    print(f"  trainable_params = {row['trainable_params']}")
    print(f"  trainable_pct = {row['trainable_pct']:.4f}%")
    print(f"  fine_tune_minutes = {row['fine_tune_minutes']:.2f}")
    print()


vanilla_phi2:
  trainable_params = 263101440
  trainable_pct = 17.2935%
  fine_tune_minutes = 0.00

default_lora:
  trainable_params = 20971520
  trainable_pct = 1.3597%
  fine_tune_minutes = 47.83

